# 08. M2 con los corpus peruano y boliviano

Se continúa desde M1, que ya fue ajustado con SMPS. En esta etapa se realizan tres épocas con la partición boliviana de entrenamiento, usando `texto_procesado` y `etiqueta_humana`. Se usa la partición boliviana de ajuste guardada.

La corrida se elige en `corrida.json`. Las evaluaciones reproducidas se guardan en `salidas/<ejecución>/evaluacion`.

Si abres este cuaderno directamente en Colab, ejecuta su primera celda y autoriza Google Drive cuando se solicite. Para recorrer toda la secuencia de una vez, usa `INICIAR_EN_COLAB.ipynb`.

In [ ]:
# Inicio al abrir este cuaderno directamente en Google Colab.
from pathlib import Path

if not Path("corrida.json").is_file():
    import os
    import subprocess
    import sys

    try:
        from google.colab import drive
    except ImportError as error:
        raise FileNotFoundError("En Jupyter, abre la carpeta que contiene corrida.json.") from error

    proyecto_drive = Path("/content/drive/MyDrive/Tesis_RoBERTuito_M0_M1_M2")
    if not (proyecto_drive / "corrida.json").is_file():
        drive.mount("/content/drive", timeout_ms=180000)
    if not (proyecto_drive / "corrida.json").is_file():
        raise FileNotFoundError("Busca Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

    os.chdir(proyecto_drive)
    sys.path.insert(0, str(proyecto_drive))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(proyecto_drive / "requirements-cuadernos.txt")],
        check=True,
    )
    print("Carpeta de la tesis conectada:", proyecto_drive)

import json
from pathlib import Path

RAIZ = Path.cwd()
CORRIDA = json.loads((RAIZ / "corrida.json").read_text(encoding="utf-8"))["corrida"]
EPOCAS = 3
SALIDAS = RAIZ / 'modelos' / CORRIDA

In [ ]:
from rutas import carpeta_salidas

from entrenamiento import leer_bolivia, ajustar

validada = carpeta_salidas(RAIZ) / "TABLAS/05_clasificacion/clasificacion_asistida_v5_etiquetada.xlsx"
datos, procedencia = leer_bolivia(RAIZ, anotacion=validada if validada.is_file() else None)

In [ ]:
destino = SALIDAS / "M2"
if (destino / "ejecucion.json").is_file():
    print("M2 guardado:", destino)
else:
    ajustar('M2', SALIDAS / 'M1', destino, datos, procedencia, epocas=EPOCAS)

## Seguimiento del ajuste de M2

El ajuste con el corpus boliviano usa **entropía cruzada multiclase**. Para una publicación,
la pérdida es el negativo del logaritmo de la probabilidad que el modelo asigna
a su etiqueta de referencia. PyTorch calcula esta pérdida al recibir los textos
tokenizados y sus etiquetas en `ajustar`. El entrenador registra un promedio cada
50 pasos de optimización.

La figura muestra esos promedios a lo largo de las tres épocas. Las líneas
verticales marcan el final de la primera y segunda época. El gráfico corresponde
al entrenamiento. La evaluación sobre las 454 publicaciones aparece en la
sección siguiente.

Para ver la curva de los pesos guardados, puedes ejecutar esta celda directamente.
Usa `CORRIDA_SEGUIMIENTO` para indicar qué ejecución se quiere mostrar.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

CORRIDA_SEGUIMIENTO = json.loads(Path("corrida.json").read_text(encoding="utf-8"))["corrida"]
modelo = "M2"
carpeta = Path("modelos") / CORRIDA_SEGUIMIENTO / modelo
datos = pd.read_csv(carpeta / "perdida_entrenamiento.csv")
ejecucion = json.loads((carpeta / "ejecucion.json").read_text(encoding="utf-8"))
pasos_totales = ejecucion["pasos_optimizacion"]
epocas = int(ejecucion["epocas"])
pasos_epoca = pasos_totales // epocas

fig, ax = plt.subplots(figsize=(8.5, 4.3))
fig.patch.set_edgecolor("#667480")
fig.patch.set_linewidth(1)
fig.suptitle(f"Pérdida de entrenamiento de {modelo}", fontsize=13, fontweight="bold")
ax.plot(datos["paso"], datos["perdida"], "o-", color="#245b93", markersize=4)
for epoca in range(1, epocas):
    ax.axvline(epoca * pasos_epoca, color="#7b8794", linestyle="--", linewidth=1)
ax.set(xlabel="Pasos de optimización", ylabel="Entropía cruzada media",
       xlim=(0, pasos_totales * 1.02), ylim=(0, datos["perdida"].max() * 1.12))
ax.grid(axis="y", alpha=0.22)
superior = ax.secondary_xaxis("top")
superior.set_xticks([n * pasos_epoca for n in range(1, epocas + 1)],
                   [str(n) for n in range(1, epocas + 1)])
superior.set_xlabel("Época completada")
fig.text(0.06, 0.025, f"Fuente. Registro de ajuste de {modelo} ({CORRIDA_SEGUIMIENTO}).", fontsize=8)
fig.tight_layout(rect=(0.03, 0.07, 0.97, 0.94))
plt.show()

## Evaluación de M2

Esta sección se puede ejecutar desde aquí con los pesos guardados. Usa las **454 publicaciones**
de la partición boliviana marcadas como `validacion` en el archivo del split de 1.809/454.
Para la corrida experimental sirven como conjunto de evaluación.

Cada publicación aporta un voto. Se usa su `texto_procesado`, se aplica
`preprocess_tweet(lang='es')` y se tokeniza en tramos de hasta 128 tokens con 32 de
solapamiento. Se promedian las probabilidades de los tramos y se toma la clase de mayor
probabilidad. La referencia es `etiqueta_humana`.

In [ ]:
from rutas import carpeta_salidas

import json
from pathlib import Path

import pandas as pd

from evaluar import evaluar_modelo, mostrar_cuadro, resumir

CORRIDA_EVALUADA = json.loads(Path("corrida.json").read_text(encoding="utf-8"))["corrida"]
predicciones = evaluar_modelo(Path.cwd(), CORRIDA_EVALUADA, "M2",
    directorio_evaluacion=carpeta_salidas(Path.cwd()) / "evaluacion",
)

La matriz de confusión (véase Cuadro 1) cuenta publicaciones por etiqueta humana (fila)
y predicción (columna). La diagonal muestra los aciertos. El Cuadro 2 muestra la
exactitud (*accuracy*), que divide los aciertos entre las 454 publicaciones, y el F1
macro, que promedia el F1 de las tres clases con el mismo peso para cada una.

In [ ]:
metricas, matriz = resumir(predicciones)
mostrar_cuadro("Cuadro 1. Matriz de confusión de M2", matriz)
mostrar_cuadro("Cuadro 2. Exactitud y F1 macro de M2", pd.DataFrame([metricas], index=["M2"]))
print(f"Aciertos: {matriz.to_numpy().trace()} de {len(predicciones)}")